# NIH Chest X-ray - Model Evaluation

This notebook evaluates the models. The resulting file is stored.

## Contents:
1. Environment Setup & Imports
2. Experiment Configuration & Reproducibility
3. Device Configuration
4. Dataset Loading
5. Model Loading
6. Inference on Test Set
7. Threshold Calibration
8. Prediction Binarization
9. Model Evaluation
10. Results Saving
11. Summary Metrics

## 1. Imports & Setup

This section imports required libraries and evaluation utilities. It ensures reproducibility and loads the trained models for inference and evaluation.

In [1]:
import torch
import numpy as np
import json
from pathlib import Path

import config
from dataset import get_dataloaders
from models import build_model
from evaluate import (
    collect_predictions,
    apply_thresholds,
    calibrate_thresholds,
    evaluate_model
)
from utils import seed_everything

## 2. Reproducibility & Experiment Selection

This section sets the experiment name and ensures reproducible evaluation conditions.

In [2]:
seed_everything()

EXPERIMENT_NAME = "exp_01_densenet_baseline"
config.set_experiment(EXPERIMENT_NAME)

print("Experiment:", config.EXPERIMENT_NAME)

Experiment: exp_01_densenet_baseline


## 3. Device Setup

This section selects the computation device used for evaluation.

In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

device

device(type='mps')

## 4. Load Dataset

This section loads the test DataLoader using the same preprocessing pipeline as training.

In [4]:
train_loader, val_loader, test_loader = get_dataloaders()

print("Test samples:", len(test_loader.dataset))

[dataset] Loading metadata ...
[dataset] Subset mode: 1002 train+val, 202 test images
[dataset] Train/val patient overlap: 0
[dataset] Split sizes - train: 894, val: 108, test: 202
Test samples: 202


## 5. Load Trained Model

This section loads the trained model checkpoint that will be evaluated on unseen test data.

In [5]:
model_name = "densenet121"

model = build_model(model_name, pretrained=False).to(device)

ckpt_path = config.CHECKPOINT_DIR / f"{model_name}_best.pt"
model.load_state_dict(torch.load(ckpt_path, map_location=device))

model.eval()

print("Loaded:", ckpt_path)

Loaded: /Users/vesco/Documents/Projects/xray-master/outputs/exp_01_densenet_baseline/checkpoints/densenet121_best.pt


## 6. Run Inference (Test Set Predictions)

This section runs inference on the full test set and collects raw probabilities and ground truth labels.

In [8]:
labels, probs = collect_predictions(model, test_loader, device)

print("Labels shape:", labels.shape)
print("Probs shape:", probs.shape)

Labels shape: (202, 15)
Probs shape: (202, 15)


## 7. Threshold Calibration (Validation Set)

This section calibrates optimal classification thresholds per class using the validation set to improve decision-making in imbalanced multi-label classification.

In [6]:
thresholds = calibrate_thresholds(model, val_loader, model_name)
thresholds

[evaluate] Calibrating thresholds on the validation set (108 samples) ...
[evaluate] Thresholds saved -> /Users/vesco/Documents/Projects/xray-master/outputs/exp_01_densenet_baseline/results/densenet121_thresholds.json


array([0.45, 0.5 , 0.27, 0.39, 0.36, 0.5 , 0.5 , 0.5 , 0.44, 0.5 , 0.35,
       0.5 , 0.31, 0.5 , 0.65], dtype=float32)

## 8. Apply Thresholds to Test Set

This section converts probability outputs into binary predictions using calibrated thresholds.

In [9]:
preds = apply_thresholds(probs, thresholds)

print("Preds shape:", preds.shape)

Preds shape: (202, 15)


## 9. Full Model Evaluation

This section computes comprehensive evaluation metrics including AUROC, AUPRC, precision, recall, F1-score, calibration metrics, and per-class performance.

In [10]:
results = evaluate_model(
    model=model,
    test_loader=test_loader,
    model_name=model_name,
    thresholds=thresholds
)

[evaluate] Running inference on the test set (202 samples) ...
[evaluate] Computing calibrated metrics ...

  Evaluation Results - densenet121
Class                     Thr    AUROC    AUPRC    Prec  Recall      F1  Support
--------------------------------------------------------------------------------
Atelectasis              0.45   0.5205   0.1462  0.1731  0.6429  0.2727       28
Cardiomegaly             0.50   0.7033   0.0476  0.0000  0.0000  0.0000        4
Consolidation            0.27   0.4662   0.0894  0.0909  0.8947  0.1650       19
Edema                    0.39   0.7429   0.1566  0.1500  0.5625  0.2368       16
Effusion                 0.36   0.4057   0.1141  0.1378  1.0000  0.2422       27
Emphysema                0.50      N/A      N/A  0.0000  0.0000  0.0000        0
Fibrosis                 0.50   0.3775   0.0130  0.0000  0.0000  0.0000        2
Hernia                   0.50      N/A      N/A  0.0000  0.0000  0.0000        0
Infiltration             0.44   0.5345   0.3821

## 10. Save Evaluation Summary

This section saves evaluation results in JSON format for later comparison across models and experiments.

In [11]:
summary_path = config.RESULTS_DIR / f"{model_name}_evaluation.json"

with open(summary_path, "w") as f:
    json.dump(results, f, indent=2)

print("Saved:", summary_path)

Saved: /Users/vesco/Documents/Projects/xray-master/outputs/exp_01_densenet_baseline/results/densenet121_evaluation.json


## 11. Top Metrics

This section prints key summary metrics for quick interpretation of model performance.

In [12]:
print("Macro AUROC:", results["macro"]["auroc"])
print("Macro AUPRC:", results["macro"]["auprc"])
print("Macro F1:", results["macro"]["f1"])
print("Hamming Loss:", results["macro"]["hamming_loss"])

Macro AUROC: 0.5507
Macro AUPRC: 0.1562
Macro F1: 0.1387
Hamming Loss: 0.3129
